# Difference imaging and detection

**Goal:** understand why we search the *difference* image, detect candidates in it, and meet the bogus zoo our
classifier will have to sort out.

Read `docs/difference_imaging.md` alongside this notebook. We only use a **training** night here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from asthunt.config.load_config import load_config
from asthunt.clean.load_exposure import load_exposure
from asthunt.detect.extract_sources import extract_sources
from asthunt.detect.cut_triplet_stamps import cut_triplet_stamps

paths = load_config("project")["paths"]
acquire = load_config("acquire")
detect = load_config("detect")

night = acquire["nights"]["train"][0]
science_files = sorted((paths["raw"] / "ztf" / night).glob("*sciimg.fits"))
exposure = load_exposure(science_files[0], acquire["bad_mask_bits"])
print(night, exposure["name"], "  bad pixels:", f"{exposure['bad'].mean():.1%}")

## 1. Why subtract?
Detecting on the **science** image finds every star and galaxy (tens of thousands). Almost none of them are new.
The **difference** image (science − reference) removes everything that didn't change.

In [ ]:
from asthunt.clean.estimate_background import estimate_background
import sep

# Compare at 5 sigma, where almost every detection is a real source (not a noise peak)
COMPARE_SIGMA = 5.0

background, noise = estimate_background(exposure["science"], exposure["bad"])
science_sources = sep.extract(np.ascontiguousarray(exposure["science"] - background, dtype=np.float32),
                              thresh=COMPARE_SIGMA, err=noise.astype(np.float32),
                              mask=exposure["bad"], minarea=detect["min_area_pixels"])
difference_sources_5sigma = extract_sources(exposure["difference"], exposure["bad"], COMPARE_SIGMA, detect["min_area_pixels"])
print("5-sigma sources in the science image   :", len(science_sources))
print("5-sigma sources in the difference image:", len(difference_sources_5sigma))

# The candidates we actually work with: the lower 2-sigma threshold
difference_sources = extract_sources(exposure["difference"], exposure["bad"], detect["threshold_sigma"], detect["min_area_pixels"])

In [ ]:
cut = (slice(1500, 1700), slice(1500, 1700))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key in zip(axes, ["science", "reference", "difference"]):
    data = exposure[key][cut]
    if key == "difference":
        ax.imshow(data, origin="lower", cmap="gray", vmin=-25, vmax=25)
    else:
        lo, hi = np.nanpercentile(data, [5, 99.5]); ax.imshow(data, origin="lower", cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(key)
plt.show()

## 2. The threshold surprise
Lower threshold = more detections, right? Not always:

In [ ]:
for threshold in [1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    found = extract_sources(exposure["difference"], exposure["bad"], threshold, detect["min_area_pixels"])
    print(f"threshold {threshold:3.1f} sigma: {len(found):6d} detections, median size {found['npix'].median():5.1f} px")

At 1.5σ, noise peaks touch each other and **merge into big blobs**, so we get *fewer*, larger detections, and a faint
asteroid can get swallowed by a noise blob next to it. At 2σ the blobs stay separate. We use 2σ and let the classifier
reject the noise. (The injection phase will measure whether 2σ really finds faint asteroids.)

## 3. What did we detect?
Pure noise gives a symmetric SNR distribution around 0. Real sources make a tail at high SNR.

In [ ]:
sources = difference_sources
plt.hist(sources["snr"].clip(-10, 40), bins=100)
plt.yscale("log"); plt.xlabel("SNR (3 px aperture)"); plt.ylabel("detections")
plt.show()
for limit in [3, 5, 10]:
    print(f"SNR > {limit:2d}: {(sources['snr'] > limit).sum()}")

## 4. The bogus zoo
Stamps of science / reference / difference. Look for:
- **noise**: nothing in science, faint speck in difference
- **bad subtraction**: a bright star in all three panels, with a positive/negative "dipole" in difference
- **cosmic ray / hot pixel**: a sharp spike, narrower than a star
- **real new source**: a star-shaped blob in science and difference, nothing in reference

In [ ]:
def show_stamps(stamps, titles, rows_of=6):
    n = len(stamps)
    fig, axes = plt.subplots(3, n, figsize=(2.2 * n, 6.6))
    for i in range(n):
        for channel, name in enumerate(["sci", "ref", "diff"]):
            image = stamps[i, channel]
            lo, hi = np.percentile(image, [1, 99.5])
            axes[channel, i].imshow(image, origin="lower", cmap="gray", vmin=lo, vmax=hi)
            axes[channel, i].set_xticks([]); axes[channel, i].set_yticks([])
            if i == 0:
                axes[channel, i].set_ylabel(name)
        axes[0, i].set_title(titles[i], fontsize=8)
    plt.show()


def stamps_for(rows):
    return cut_triplet_stamps(exposure["science"], exposure["reference"], exposure["difference"],
                              rows["x"].to_numpy(), rows["y"].to_numpy(), size=detect["stamp_size"])


random_rows = sources.sample(6, random_state=1)
show_stamps(stamps_for(random_rows), [f"SNR {s:.1f}" for s in random_rows["snr"]])

Random detections are mostly noise. Now the brightest ones:

In [ ]:
bright_rows = sources.sort_values("snr", ascending=False).head(6)
show_stamps(stamps_for(bright_rows), [f"SNR {s:.0f}" for s in bright_rows["snr"]])

## 5. The whole night

In [ ]:
rows = []
for path in science_files:
    one = load_exposure(path, acquire["bad_mask_bits"])
    found = extract_sources(one["difference"], one["bad"], detect["threshold_sigma"], detect["min_area_pixels"])
    rows.append({"frame": one["name"][-30:], "minutes": (one["obsjd"] - exposure["obsjd"]) * 24 * 60,
                 "detections": len(found), "snr_above_5": int((found["snr"] > 5).sum()),
                 "seeing_px": one["header"]["SEEING"]})
night_summary = pd.DataFrame(rows)
night_summary.round(1)

Roughly 8,000 candidates per frame, 5 frames. If we tried every combination of one detection per frame to look for
straight lines, that would be 8000⁵ ≈ 3 × 10¹⁹ combinations. Two things save us: a **classifier** that throws away most
noise first, and a **smart linker** that only tries combinations that make physical sense.

## Exercises
1. Find one dipole (bad subtraction) in the bright stamps. Why does it have a positive and a negative half?
2. How many detections have SNR > 5 in every frame? Are they the same objects? (Hint: compare x, y between frames.)
3. Change `min_area_pixels` to 3. What happens to the count, and what kinds of bogus get through?